### Import data (download dataset from Kaggle to tmp folder)

In [ ]:
# Download datasets
import os

os.system("Kaggle datasets download -d zusmani/paradisepanamapapers -p /tmp/paradise --unzip --force")


### Load datasets

In [ ]:
# Load the combined files into df
import pandas as pd 

for fname in ['Addresses.csv', 'Entities.csv', 'Intermediaries.csv', 'Officers.csv', 'all_edges.csv']:
    df = pd.read_csv(f'/tmp/paradise/{fname}', low_memory = False)
    print(f'{fname}: {df.shape}, {df.columns.to_list()}')
    
data_dir = '/tmp/paradise/'
addresses = pd.read_csv(f"{data_dir}Addresses.csv", low_memory = False)
entities = pd.read_csv(f"{data_dir}/Entities.csv", low_memory = False)
intermediaries = pd.read_csv(f"{data_dir}/Intermediaries.csv", low_memory = False)
officers = pd.read_csv(f"{data_dir}/Officers.csv", low_memory = False)
edges = pd.read_csv(f"{data_dir}/all_edges.csv", low_memory = False)

print("\nAll data frames loaded successfully")


In [ ]:
import os

files = ['Addresses.csv', 'Entities.csv', 'Intermediaries.csv', 'Officers.csv', 'all_edges.csv']

for fname in files:
    path = f'/tmp/paradise/{fname}'
    size = os.path.getsize(path)
    print(f'{fname:25s} : {size/1024/1024:.1f} MB')

### Data validation

In [ ]:
# 1. Preview each dataframe
for name, df in [('addresses', addresses), ('entities', entities), ('intermediaries', intermediaries), ('officers', officers), ('edges', edges)]:
    print(f'\n{name}: {df.shape}')
    print(df.columns.tolist())
    display(df.head(3))
    
# for name, df in [('addresses', addresses), ('entities', entities), ('intermediaries', intermediaries), ('officers', officers), ('edges', edges)]:
#     print(f'{name}')
#     display(df.head(3))

*From edges preview:*
- `node_1` = source node
- `node_2` = destination node  

In [ ]:
# 2. Check relationship types 
# What types of relationships exist?
print(edges['rel_type'].value_counts())

In [ ]:
# 3. Check data connection across files using key columns. In our df, we use node_id
# ('addresses', addresses), ('entities', entities), ('intermediaries', intermediaries), ('officers', officers), ('edges', edges)]
print(f"addresses node_ID: {addresses['node_id'].iloc[0]}")
print(f"entities node_ID: {entities['node_id'].iloc[0]}")
print(f"intermediaries node_ID: {intermediaries['node_id'].iloc[0]}")
print(f"officers node_ID: {officers['node_id'].iloc[0]}")

# 'edges' df use node_1 and node_2
print(f"edges node_1: {edges['node_1'].iloc[0]}")
print(f"edges node_2: {edges['node_2'].iloc[0]}")


In [ ]:
# 4. Do node ID in 'edges' match node ID in node files?
# Join all node files into one big table of the nodes, on key column = node_id
all_node_id = set(addresses['node_id']) | set(entities['node_id']) | set(intermediaries['node_id']) | set(officers['node_id'])
edges_node_id = set(edges['node_1']) | set(edges['node_2'])

# Find matching node_IDs or unique nodes
matched_node = all_node_id & edges_node_id # exist in both tables
unmatched_node = all_node_id - edges_node_id # left join tables - no match in edges

print(f'Total unique node IDs across all node files: {len(all_node_id)}')
print(f'Total unique node IDs in edges: {len(edges_node_id)}')
print(f'Matched node IDs (exist in both): {len(matched_node)}')
print(f'Unmatched (in node files but not in edges): {len(unmatched_node)}')

In [ ]:
# Check for duplicate edges - same node_id with multiple relationship type
# This will decide the type of graphs we need later
duplicate_edges = edges[edges.duplicated(subset=['node_1', 'node_2'])]
print(len(edges))
print(len(duplicate_edges))

*Analysis*
- All node IDs in 'edges' file exist in node files, so complete info for graph.
- There are 208 nodes without edges or incomplete information. These orphans are very small % compared to the matched. So we can safely ignore these cases.
- Nodes have multiple edges (76,320 duplicated edges), so the graph will require a MultiDiGraph type of graph.

### Graph (NetworkX)

*Pipeline*
- Data in CSV files -> Data Validation -> Data Preparation -> Graph building -> Analyse -> Visualise


In [ ]:
# Prepare data for graph building
# Add node_type column to each df for easy identification of entity
addresses['node_type'] = 'address'
entities['node_type'] = 'entity'
intermediaries['node_type'] = 'intermediary'
officers['node_type'] = 'officer'

In [ ]:
# Standardise the label column so all node files use 'name' as label column except for addresses
# 'Addresses' use 'address' so we will rename this to 'name' to match the other files

addresses = addresses.rename(columns = {'address': 'name'})
#check renaming
#addresses.columns.tolist()
print(f'addresses columns: {addresses.columns.tolist()}')

In [ ]:
# Combine all node files into one master node df
# Keep columns needed for graph
    # node_id - unique identifier
    # name - display label
    # node_type - type of node (entity, officer, intermediary, address)
    
master_nodes = pd.concat([
    addresses[['node_id', 'name', 'node_type']],
    entities[['node_id', 'name', 'node_type']],
    intermediaries[['node_id', 'name', 'node_type']],
    officers[['node_id', 'name', 'node_type']]
], ignore_index=True)

print(master_nodes.shape)
master_nodes['node_type'].value_counts()


### Create Graph

In [ ]:
# Create an empty MultiDiGraph with no edges and no nodes

import networkx as nx

G = nx.MultiDiGraph()

### Add nodes

In [ ]:
# Add nodes to the graph using the master_nodes df
# .add_nodes_from() takes list of 2 tuples (node_id, {attributes dictionary})
# Build attributes as list comprehension from master_nodes df -> better for larger dataset
G.add_nodes_from([
    (row['node_id'], { # unique ID
        'name': row['name'], # display label
        'node_type': row['node_type'] # entity type
        })
        for _, row in master_nodes.iterrows()
])
    
print(f'{G.number_of_nodes():,}')


*Notes*
- Number of nodes is 1,040,535. Not matching the number of nodes in the master_nodes df (1,041,674). In the master_nodes df, there are duplicates!

In [ ]:
# Check for duplicate node_ids in master_nodes
duplicate_nodes = master_nodes[master_nodes.duplicated(subset=['node_id'])]
print(f'Duplicate node_ids : {len(duplicate_nodes)}')
print(duplicate_nodes.head(10))

*Notes*
- Interesting...there are 1139 duplicated node_id (or appear in more than 1 node file), and all of them are officers.
- 1,139 duplicates out of 1,041,674 rows and NetworkX keep the first occurence. 
- The duplicate is 0.1% of total nodes so it is negligible at this stage but investigate later.

### Add edges

In [ ]:
# Add edges to graph
# .add_edges_from() with 3-tuples: ([[node1, node2], {'attributes dictionary}])
# Include all edges attributes from the edges df
G.add_edges_from([
    (row['node_1'], row['node_2'], {
        'rel_type': row['rel_type'],
        'sourceID': row['sourceID'],
        'valid_until': row['valid_until'],
        'start_date': row['start_date'],
        'end_date': row['end_date']})
    for _, row in edges.iterrows()
])

print(f'{G.number_of_edges():,}')


In [ ]:
# Check one node & edge added
sample_node = list(G.nodes(data = True))[0]
print(sample_node)

sample_edge = list(G.edges(data = True))[0]
print(sample_edge)

In [ ]:
# Check node types of node_1 and node_2 for first 5 edges
print('Confirm edge direction:')
for _, row in edges.head(3).iterrows():
    type_node_1 = G.nodes[row['node_1']]['node_type']
    type_node_2 = G.nodes[row['node_2']]['node_type']
    print(f'node_1 ({type_node_1}) to {row['rel_type']} to node_2 ({type_node_2})')


*Analysis*
- Graph successfully built. Graph has 1,040,535 nodes and 1,535,552 edges.
- Node contained all attributes assigned. Minor data quality issues observed as some address names contain tab characters (e.g address contain -\t...) from raw data.
- Edge stored correctly. NaN values in date fields are expected, not all relationships have date information.

*Edge direction confirmed:*
- `node_1` = source (e.g. intermediary, officer)
- `node_2` = destination (e.g. entity)
- Makes logical sense: intermediary → sets up → entity. Direction is consistent across all sample edges meaning graph was built correctly.


### Graph Analysis/Network Analysis
- Get overview of graph structure:
   - how connected the network is to understand the shape of the network
   - what does the distribution look like to find the key players
   - are there any isolated clusters?

***Basic Graph Statistics***

In [ ]:
# Let's do basic graph analysis
# Equivalent to pandas df.describe()
# Check graph structure built above
print(f'Graph type: {type(G).__name__}')
print(f'Total nodes: {G.number_of_nodes(): ,}')
print(f'Total edges: {G.number_of_edges(): ,}')
print(f'Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}') # find out how many connections each node has in average

In [ ]:
# Check connected components in the graph for:
   # connected components are group of nodes that are connected to each other
   # isolated nodes i.e nodes without edges are nodes in their own component 

# Convert graph G to undirected type to simplify clustering
G_undirected = G.to_undirected()

# Find connected components
components = list(nx.connected_components(G_undirected))
component_sizes = sorted([len(c) for c in components], reverse = True)

print(f'Total connected components: {len(components):,}')
print(f'Largest component: {component_sizes[0]:,} nodes')
print(f'Second largest component: {component_sizes[1]:,} nodes')
print(f'Isolated nodes (size = 1): {sum(1 for s in component_sizes if s == 1):,}')

*Analysis*
- Basic graph stats show:
   - Network is quite sparse. Each node average less than 3 connections (2.95).
   - Statistics suggest most nodes have few connections (low degree), which is reflective of real-world networks.
   - My graph suggest to have a few nodes with high degree connections, which warrant investigation.

- Connected components show:
   - There are 10,633 clusters in the network.
   - There is one massive network with 942,172 nodes. This is the main networkt where most activity is interconnected.
   - Second largest is less than 2,000 nodes. Relatively small compared to the main one and likely not linked to the main.
   - The isolated nodes are small.
   - For deeper analysis, focus on the main network as it contains the most activity.

***Network distribution***

In [ ]:
# Check how many connections each node have and visualise the network distribution

import matplotlib.pyplot as plt
%matplotlib inline 

# Get degree for every node
degree_values = [degree for node, degree in G.degree()] # G.degree() returns (node id, degree) pairs, we only want the degree values

In [ ]:
# Check in-degree and out-degree separately
print(f'Total edges          : {G.number_of_edges():,}')
print(f'Sum of in-degrees    : {sum(d for n, d in G.in_degree()):,}')
print(f'Sum of out-degrees   : {sum(d for n, d in G.out_degree()):,}')
print(f'Sum of total degrees : {sum(d for n, d in G.degree()):,}')

In [ ]:
# Plot degree distribution
plt.figure(figsize = (10, 6))
plt.hist(degree_values, bins=50, color='steelblue', edgecolor='white')
plt.title('KG Paradise degree distribution')
plt.xlabel('Degree (number of connections)')
plt.ylabel('Number of nodes (log scale)')
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print(f'Min degree: {min(degree_values):,}')
print(f'Max degree: {max(degree_values):,}')
print(f'Mean degree: {sum(degree_values)/len(degree_values):.2f}')
 

In [ ]:
# Let's find who is the node with the most number of connection :D
# Find the most connected node
max_degree_node = max(G.degree(), key = lambda x: x[1])

print('Most connected node:')
print(f'node id: {max_degree_node[0]}')
print(f'degree: {max_degree_node[1]:,}')
print(f'name: {G.nodes[max_degree_node[0]]['name']}')
print(f'node type: {G.nodes[max_degree_node[0]]['node_type']}')

*Analysis*
- Interesting finding. The most connected node is an 'address', not a person or a company with 37,338 connections.
- This suggest there are 37,338 entities registered to this address, and this is a red flag as this could be an address to a ghost company which is used by many entities for their offshore activities! Red flag!!!



***Degree of centrality***

In [ ]:
# Find top 10 most connected nodes
top_10_nodes = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]

print('Top 10 most connected nodes:')
for rank, (node_id, degree) in enumerate(top_10_nodes, start=1):
    name      = G.nodes[node_id]['name']
    node_type = G.nodes[node_id]['node_type']
    print(f'{rank}. {name} | {node_type} | degree: {degree:,}')

*Analysis*
- Top 10 most connected nodes are:
  - Corporate service providers (Portcullis TrustNet)
  - Known Panama Papers intermediaries (Mossack Fonseca)
  - Major financial institutions acting as trustees (UBS, Credit Suisse)
  - Mass registered addresses in BVI and London

***PageRank***


In [ ]:
# Rank nodes by the importance of nodes pointing TO them
# High PageRank means pointed to by other important nodes
# Set alpha = 0.85 as standard damping factor

pagerank = nx.pagerank(G, alpha=0.85)

# Get top 10 nodes by PageRank score
top_10_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]

print('Top 10 nodes by PageRank:')
for rank, (node_id, score) in enumerate(top_10_pagerank, start=1):
    name      = G.nodes[node_id]['name']
    node_type = G.nodes[node_id]['node_type']
    print(f'{rank}. {name} | {node_type} | score: {score:.6f}')

*Analysis*
- All top 10 nodes are addresses, not people or companies.
- Portcullis TrustNet ranks #1 in both degree centrality and PageRank suggesting this is the most structurally important node in the entire network.
- Address registered in Hong Kong dominates the list (4 out of 10).

In [ ]:
# Degree of centrality and PageRank give the top 10 most connected node and top 10 most important nodes.
# Let's check which entities actually pointing to the top 10 addreses in PageRank
# For each of the top 10 PageRank address, how many entities and who is the most frequent entity pointing to the address? 
for node_id, score in top_10_pagerank:
    name = G.nodes[node_id]['name'][:50] # truncate name to 50 characters
    in_edges = edges[edges['node_2'] == node_id]
    count = len(in_edges)
    top_source_id = in_edges['node_1'].value_counts().index[0]
    top_source = G.nodes[top_source_id]['name'][:50]
    
    print(f'{count:,} entities point to: {name}')
    print(f'Top source: {top_source}')
    print()


### Visualise KG Paradise

In [ ]:
from pyvis.network import Network

# Get top 50 most connected nodes (we have more than 1 million network, cannot visualise all in the browser)
top_50_nodes = sorted(G.degree(), key = lambda x: x[1], reverse=True)[:50] # return top 50 (node_id, degree) pair in order highest to lowest
top_50_ids = [node_id for node_id, degree in top_50_nodes]

print(f'Top 50 nodes selected:')
print(f'Degree range: {top_50_nodes[-1][1]:,} to {top_50_nodes[0][1]:,}') # get degree value [1] for node in 50th [-1] and 1st [0]

In [ ]:
# Create subgraph of the top 50 nodes
# subgraph() only returs the nodes and edges between them

subgraph = G.subgraph(top_50_ids).copy()

print(f'Subgraph nodes: {subgraph.number_of_nodes()}')
print(f'Subgraph edges: {subgraph.number_of_edges()}')

*Analysis* 
- Only 3 edges !!! This means the major nodes are not directly connected to each other. Each connect to many smaller nodes but nod to each other, probably trying to avoid suspicion :D
-Visualising the top 50 graph won't look interesting since there's only 3 edges to show! 
- Let's pick 1 node (the top 1 Portculis TrustNet) and visualise its edges.

In [ ]:
import random
random.seed(42)

# Get all nodes directly connected to Portcullis TrustNet (top node)
portcullis_id = top_50_ids[0]  # node with highest degree

# Get immediate neighbours
neighbours = list(G.neighbors(portcullis_id))
print(f'Portcullis TrustNet has {len(neighbours):,} direct neighbours')

# Use whichever is smaller — 50 or actual number of neighbours
sample_size      = min(50, len(neighbours))
sample_neighbours = random.sample(neighbours, sample_size)

# Build subgraph
viz_nodes = [portcullis_id] + sample_neighbours
subgraph  = G.subgraph(viz_nodes).copy()

print(f'Subgraph nodes : {subgraph.number_of_nodes()}')
print(f'Subgraph edges : {subgraph.number_of_edges()}')

*Analysis*
Interesting...0 direct neighbours means Portcullis TrustNet is the destination node (node_2), not the source (node_1). So we need to check predecessors (source) instead of neighbours (destination)...

In [ ]:
# G.neighbors() only returns outgoing connections (node_1 → node_2)
# Portcullis TrustNet is node_2 (destination) so we need predecessors

portcullis_id = top_50_ids[0]

# predecessors = nodes pointing TO this node (Portcullis TrustNet)
predecessors = list(G.predecessors(portcullis_id))
print(f'Portcullis TrustNet has {len(predecessors):,} predecessors')

*Analysis*
- Portcullis TrustNet is pure destination node! Everything poiints to it! So, this is the enabler of all the shell companies.

### Built subgraph for visual

In [ ]:
# Take a sample of 50 source nodes for visualisation
sample_size = min(50, len(predecessors)) 
sample_predecessors = random.sample(predecessors, sample_size) # choose random 50 from the 37K ++ predecessors pointing to Portcullis TrustNet

# Build visual of Portcullis TrustNet + 50 predecessors
viz_nodes = [portcullis_id] + sample_predecessors # combine Portcullis node_id with the 50 predecessors node_ids into one list
subgraph = G.subgraph(viz_nodes).copy() # create the subgraph

print(f'Subgraph nodes: {subgraph.number_of_nodes()}')
print(f'Subgraph edges: {subgraph.number_of_edges()}')

### Build the PyVis visualisation

In [ ]:
from pyvis.network import Network

# Create interactive network
net = Network(height='1000px', width='150%', directed=True, notebook=True, cdn_resources='in_line')

# Add nodes with colours by node type
node_colours = {
    'entity': 'red',
    'officer': 'blue',
    'intermediary': 'orange',
    'address': 'green'
}

for node_id in  subgraph.nodes():
    name = subgraph.nodes[node_id]['name'][:40]
    node_type = subgraph.nodes[node_id]['node_type']
    colour = node_colours.get(node_type, 'grey')
    net.add_node(node_id, label=name, color= colour, title = f'{node_type}: {name}')
    
# Add edges
for src, tgt, data in subgraph.edges(data = True):
    net.add_edge(src, tgt, title=data.get('rel_type', ''))
    
# Save as HTML
net.save_graph('KG_Paradise_Visual.html')
print('Visual saved -> KG_Paradise_Visual.html')

In [ ]:
# from IPython.display import HTML
# display(HTML('KG_Paradise_Visual.html'))

In [ ]:
import os
os.system('open KG_Paradise_Visual.html')  # opens in default browser on Mac

*Visualisation*
- Star shaped network with Portcullis TrustNet (green) at centre.
- 50 shell companies (red) all registered at the same address.
- One officer node (blue "Liao Sheng Man") visible in the network as part of the sampled predecessors (source: node 1). This node can be investigated further!
- Classic mass-registration pattern — red flag in fincrime analysis,

### Summary 

In [ ]:
# Notebook Summary
print(f'Nodes              : {G.number_of_nodes():,}')
print(f'Edges              : {G.number_of_edges():,}')
print(f'Top node           : Portcullis TrustNet — #1 degree and PageRank')


*Key finding:*
Portcullis TrustNet BVI is the most connected and most influential node, with 37,338 shell companies registered to this one address in the British Virgin Islands.